In [2]:

import random
import requests

# 현재 실행 중인 FastAPI 서버
BASE_URL = "http://127.0.0.1:8000"

# 5회차에서 구현한 네 가지 Tool과 일반 답변
TOOL_OPTIONS = {
    "1": "calculator",
    "2": "text_length",
    "3": "current_time",
    "4": "data_summary",
    "5": None
}

# 게임 문제: (질문, 기대되는 Tool)
MISSIONS = [
    ("125 곱하기 8에 30을 더하면?", "calculator"),
    ("'AI Agent'의 글자 수를 계산해줘.", "text_length"),
    ("지금 한국 날짜와 시간은?", "current_time"),
    ("판매액 [10,20,30]의 평균과 합계를 알려줘.", "data_summary"),
    ("LLM과 AI Agent의 차이를 한 문장으로 설명해줘.", None)
]

# 서버 상태 검사
try:
    health = requests.get(BASE_URL + "/health", timeout=10)
    health.raise_for_status()

    if not health.json().get("cuda_available"):
        raise RuntimeError("CUDA GPU가 연결되지 않았습니다.")

except Exception as e:
    raise RuntimeError("FastAPI 서버와 GPU 상태를 확인하세요.") from e

human_score = 0
ai_score = 0

print("=" * 50)
print("     🎮 인간 vs AI Agent 챌린지 🎮")
print("=" * 50)

# 문제를 무작위 순서로 출제
for round_no, (question, expected_tool) in enumerate(
    random.sample(MISSIONS, len(MISSIONS)), start=1
):
    print(f"\n🎯 ROUND {round_no}")
    print("문제:", question)

    print("\n어떤 Tool을 사용해야 할까요?")
    for key, tool in TOOL_OPTIONS.items():
        print(f"{key}. {tool if tool else 'Tool 불필요'}")

    choice = input("\n정답 번호 입력 (1~5): ").strip()

    while choice not in TOOL_OPTIONS:
        choice = input("1~5 중에서 선택하세요: ").strip()

    human_prediction = TOOL_OPTIONS[choice]

    # 학생 정답 판정
    if human_prediction == expected_tool:
        human_score += 1
        print("✅ 학생 정답!")
    else:
        print("❌ 학생 오답!")

    # 실제 Qwen3-8B Agent에게 질문
    print("\n🤖 AI Agent가 판단하는 중...")

    try:
        response = requests.post(
            BASE_URL + "/agent",
            json={"query": question},
            timeout=240
        )
        response.raise_for_status()
        result = response.json()

        actual_tool = result.get("tool_name")
        status = result.get("status")

        if status == "success" and actual_tool == expected_tool:
            ai_score += 1
            print("✅ AI Agent 정답!")
        else:
            print("❌ AI Agent 오답 또는 실행 실패!")

        print("기대 Tool:", expected_tool or "Tool 불필요")
        print("AI 선택:", actual_tool or "Tool 미사용")
        print("실행 상태:", status)
        print("AI 답변:", result.get("final_answer", "")[:250])

    except requests.RequestException as e:
        print("⚠️ API 호출 실패:", e)

    print(f"\n현재 점수 → 학생 {human_score} : AI {ai_score}")

print("\n" + "=" * 50)
print("🏆 최종 결과")
print(f"학생: {human_score}점 / 5점")
print(f"AI Agent: {ai_score}점 / 5점")

if human_score > ai_score:
    print("🎉 학생 승리!")
elif human_score < ai_score:
    print("🤖 AI Agent 승리!")
else:
    print("🤝 무승부!")


     🎮 인간 vs AI Agent 챌린지 🎮

🎯 ROUND 1
문제: 판매액 [10,20,30]의 평균과 합계를 알려줘.

어떤 Tool을 사용해야 할까요?
1. calculator
2. text_length
3. current_time
4. data_summary
5. Tool 불필요



정답 번호 입력 (1~5):  1


❌ 학생 오답!

🤖 AI Agent가 판단하는 중...
✅ AI Agent 정답!
기대 Tool: data_summary
AI 선택: data_summary
실행 상태: success
AI 답변: 평균은 20.0이고, 합계는 60입니다.

현재 점수 → 학생 0 : AI 1

🎯 ROUND 2
문제: 지금 한국 날짜와 시간은?

어떤 Tool을 사용해야 할까요?
1. calculator
2. text_length
3. current_time
4. data_summary
5. Tool 불필요



정답 번호 입력 (1~5):  2


❌ 학생 오답!

🤖 AI Agent가 판단하는 중...
❌ AI Agent 오답 또는 실행 실패!
기대 Tool: current_time
AI 선택: Tool 미사용
실행 상태: error
AI 답변: Agent 결정을 처리하지 못했습니다.

현재 점수 → 학생 0 : AI 1

🎯 ROUND 3
문제: 'AI Agent'의 글자 수를 계산해줘.

어떤 Tool을 사용해야 할까요?
1. calculator
2. text_length
3. current_time
4. data_summary
5. Tool 불필요



정답 번호 입력 (1~5):  2


✅ 학생 정답!

🤖 AI Agent가 판단하는 중...
✅ AI Agent 정답!
기대 Tool: text_length
AI 선택: text_length
실행 상태: success
AI 답변: 'AI Agent'의 글자 수는 8입니다.

현재 점수 → 학생 1 : AI 2

🎯 ROUND 4
문제: 125 곱하기 8에 30을 더하면?

어떤 Tool을 사용해야 할까요?
1. calculator
2. text_length
3. current_time
4. data_summary
5. Tool 불필요



정답 번호 입력 (1~5):  1


✅ 학생 정답!

🤖 AI Agent가 판단하는 중...
✅ AI Agent 정답!
기대 Tool: calculator
AI 선택: calculator
실행 상태: success
AI 답변: 125 곱하기 8에 30을 더하면 1030이다.

현재 점수 → 학생 2 : AI 3

🎯 ROUND 5
문제: LLM과 AI Agent의 차이를 한 문장으로 설명해줘.

어떤 Tool을 사용해야 할까요?
1. calculator
2. text_length
3. current_time
4. data_summary
5. Tool 불필요



정답 번호 입력 (1~5):  4


❌ 학생 오답!

🤖 AI Agent가 판단하는 중...
✅ AI Agent 정답!
기대 Tool: Tool 불필요
AI 선택: Tool 미사용
실행 상태: success
AI 답변: LLM은 대규모 언어 모델로 텍스트 생성 및 이해에 특화되어 있지만, AI Agent는 특정 작업을 수행하기 위한 목표 지향적 시스템이다.

현재 점수 → 학생 2 : AI 4

🏆 최종 결과
학생: 2점 / 5점
AI Agent: 4점 / 5점
🤖 AI Agent 승리!
